# Coffee ordering assistant — QLoRA fine-tune

Order of operations matters here: the base model is measured **before** any
adapter exists, so the final table has a real "before" column. Loss going down
is not evidence; beating the base model on held-out data is.

| | |
|---|---|
| Base | `unsloth/Qwen3-4B-Instruct-2507-bnb-4bit` |
| Method | QLoRA, rank 16 |
| Hardware | Colab free T4 (16 GB, fp16 — T4 has no bf16) |
| Output | LoRA adapter pushed to the Hub, never merged |

Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
!nvidia-smi -L
%pip install -q unsloth
%pip install -q --no-deps --upgrade "trl>=0.9" peft accelerate bitsandbytes

## 1. Data and scoring code

Both come from the repo, so the notebook stays thin and the metrics match what CI checks.

In [ ]:
!git clone -q --branch claude/coffee-finetuned-order-system-pnendq https://github.com/eneskaya96/llm-fine-tune-rag-full-product.git repo

import sys
sys.path.append("/content/repo/scripts")

import evaluate as ev

train_records = ev.load("/content/repo/data/train.jsonl")
eval_records = ev.load("/content/repo/data/eval.jsonl")
print(f"train {len(train_records)}  eval {len(eval_records)}")

## 2. Load the base model in 4-bit

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3-4B-Instruct-2507-bnb-4bit",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)

## 3. Baseline — the base model, measured twice

`format_ok` collapses to 0% if the model is told to "emit a create_order tool
call" without being shown the schema. That measures the prompt, not the model,
so the base is scored both ways: with the training prompt as-is, and with the
tool definition appended. The fine-tuned model then has to beat the *armed*
baseline to have proven anything.

In [ ]:
def generate(records, tool_schema=False, batch_size=8, max_new_tokens=160):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.generation_config.max_length = None   # silences a max_length warning

    outputs = []
    for start in range(0, len(records), batch_size):
        chunk = records[start:start + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                ev.prompt_messages(r, tool_schema=tool_schema),
                tokenize=False, add_generation_prompt=True)
            for r in chunk
        ]
        batch = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            generated = model.generate(
                **batch,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        cut = batch.input_ids.shape[1]
        outputs += tokenizer.batch_decode(generated[:, cut:], skip_special_tokens=True)
        print(f"  {min(start + batch_size, len(records))}/{len(records)}", end="\r")
    return outputs


# Two baselines. The training prompt names create_order but never defines it --
# the tuned model learns the schema from weights, a base model cannot. Scoring
# only the first would measure our prompt's omission, not the model.
base_plain_out = generate(eval_records)
base_plain = ev.summarise(eval_records, base_plain_out)
print(ev.format_table(base_plain, "BASE (menu prompt only)"), "\n")

base_armed_out = generate(eval_records, tool_schema=True)
base_armed = ev.summarise(eval_records, base_armed_out)
print(ev.format_table(base_armed, "BASE (+ tool schema)"))

## 4. Attach the LoRA adapter

Rank 16 across attention and MLP projections. Higher rank overfits 800
examples; lower struggles with multi-turn tool-call discipline.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

## 5. Train

Loss is computed on assistant turns only — the model should not be learning to predict the menu we hand it.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def to_text(record):
    return {"text": tokenizer.apply_chat_template(record["messages"], tokenize=False)}

train_ds = Dataset.from_list([to_text(r) for r in train_records])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    args=SFTConfig(
        output_dir="outputs",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="linear",
        logging_steps=10,
        optim="adamw_8bit",
        fp16=True,          # T4 has no bf16
        seed=42,
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()

## 6. Measure again on the same held-out set

In [ ]:
tuned_out = generate(eval_records)
tuned = ev.summarise(eval_records, tuned_out)
print(ev.format_table(tuned, "FINE-TUNED (menu prompt only)"))

## 7. Before / after

This table goes in the README.

In [ ]:
rows = [("base (menu only)", base_plain), ("base (+schema)", base_armed), ("fine-tuned", tuned)]

print(f"{'metric':14}" + "".join(f"{name:>18}" for name, _ in rows))
for metric in ev.METRICS:
    line = f"{metric:14}"
    for _, summary in rows:
        value = summary["overall"][metric]
        line += f"{'n/a' if value is None else f'{value:.1%}':>18}"
    print(line)

gain = tuned["overall"]["exact_match"] - base_armed["overall"]["exact_match"]
print(f"\nexact_match vs armed baseline: {gain:+.1%}")

print("\nExample generations:")
for record, before, after in list(zip(eval_records, base_armed_out, tuned_out))[:3]:
    print(f"\n[{record['meta']['category']}] {record['messages'][-2]['content']}")
    print(f"  BASE : {before.strip()[:200]}")
    print(f"  TUNED: {after.strip()[:200]}")

## 8. Publish the adapter

Not merged: merging folds the weights into the base model, which kills the
hot-swap the serving layer depends on. ~200 MB of adapter, base stays shared.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
REPO_ID = "your-username/coffee-order-friendly"   # <- change this

model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)
print("pushed:", REPO_ID)